# DINOv3 Distillation to YOLO11l-seg

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marjanstoimcev/DINOv3distill/blob/main/dinov3_distillation.ipynb)

This notebook demonstrates how to distill knowledge from DINOv3 (ViT-S/16) to YOLO11l-seg using LightlyTrain.

**Workflow:**
1. Install dependencies
2. Download TACO dataset
3. **Pretrain** YOLO11l-seg backbone using DINOv3 distillation (unlabeled images)
4. **Fine-tune** on labeled instance segmentation task (polygon masks)

**Compare with:** `train_from_scratch.ipynb` (baseline without pretraining)

**Model:**
- YOLO11l-seg: 27.7M params, 143.0 GFLOPs

**Dataset:** TACO - Trash Annotations in Context
- 59 classes of litter/trash
- 1,499 images with polygon segmentation masks
- License: CC BY 4.0

## 1. Install Dependencies

In [ ]:
# Install lightly-train with ultralytics support
!pip install -q "lightly-train[ultralytics]" gdown

# Verify installation
import lightly_train
print(f"LightlyTrain version: {lightly_train.__version__}")

In [ ]:
# Additional imports
import os
import shutil
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Plotting style - clean, no grids
plt.style.use('default')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'font.size': 11,
})

COLORS = {'blue': '#2563eb', 'red': '#dc2626', 'green': '#059669', 'orange': '#d97706'}

## 2. Hyperparameters

Configure training hyperparameters here. Defaults based on [LightlyTrain recommendations](https://docs.lightly.ai/train/).

In [ ]:
# ============================================================
# HYPERPARAMETERS - Modify these as needed
# ============================================================

# Distillation pretraining
DISTILL_EPOCHS = 100        # LightlyTrain recommends 100-300 for DINOv2/v3
DISTILL_BATCH_SIZE = 128    # Recommended: 128-1536 (adjust for GPU memory)
DISTILL_LR = None           # None = auto-scaled based on batch size

# Fine-tuning
FINETUNE_EPOCHS = 50        # Ultralytics default for fine-tuning
FINETUNE_BATCH_SIZE = 16    # Adjust based on GPU memory
FINETUNE_LR = 0.01          # Ultralytics default
FINETUNE_LRF = 0.01         # Final LR = LR * LRF

# Common
IMAGE_SIZE = 640            # Input image size
PATIENCE = 10               # Early stopping patience
NUM_WORKERS = 8             # Data loading workers
PRECISION = "bf16-mixed"    # Mixed precision: "bf16-mixed", "16-mixed", "32"

# Model
STUDENT_MODEL = "ultralytics/yolo11l-seg.yaml"  # 27.7M params
TEACHER_MODEL = "dinov3/vits16"                  # DINOv3 ViT-S/16

print("Hyperparameters:")
print(f"  Distillation: {DISTILL_EPOCHS} epochs, batch={DISTILL_BATCH_SIZE}, lr={DISTILL_LR or 'auto'}")
print(f"  Fine-tuning:  {FINETUNE_EPOCHS} epochs, batch={FINETUNE_BATCH_SIZE}, lr={FINETUNE_LR}")
print(f"  Image size:   {IMAGE_SIZE}")
print(f"  Precision:    {PRECISION}")
print(f"  Num workers:  {NUM_WORKERS}")
print(f"  Student:      {STUDENT_MODEL}")
print(f"  Teacher:      {TEACHER_MODEL}")

## 3. Dataset Setup

**TACO: Trash Annotations in Context** - Instance segmentation dataset of litter in the wild.

59 classes including:
- Plastic bottles, cups, bags, wrappers
- Glass bottles, jars
- Metal cans, foil, bottle caps
- Paper, cardboard, cartons
- Cigarettes, food waste, and more

Dataset splits:
- Train: 1,049 images
- Valid: 299 images  
- Test: 151 images

Source: [Roboflow Universe](https://universe.roboflow.com/mohamed-traore-2ekkp/taco-trash-annotations-in-context)

In [ ]:
# Download TACO dataset from Google Drive
import gdown
import zipfile

FILE_ID = "1HZBR53rs_igtr2ZlYqsLrS0uROCcTqzl"
ZIP_PATH = "taco.zip"
DATA_PATH = Path("./data/taco")

if not DATA_PATH.exists():
    print("Downloading TACO dataset from Google Drive...")
    gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", ZIP_PATH, quiet=False)
    
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('./data')
    print("Dataset ready!")
else:
    print(f"Dataset already exists at {DATA_PATH}")

dataset_path = DATA_PATH
!ls -la ./data/taco/

In [ ]:
# Verify dataset structure
print("Dataset structure:")
print(f"  Root: {dataset_path}")

total_images = 0
for split in ['train', 'valid', 'test']:
    images_path = dataset_path / split / 'images'
    labels_path = dataset_path / split / 'labels'
    
    if images_path.exists():
        num_images = len(list(images_path.glob('*.jpg'))) + len(list(images_path.glob('*.JPG'))) + len(list(images_path.glob('*.png')))
        total_images += num_images
        print(f"  {split}/images: {num_images} images")
    
    if labels_path.exists():
        num_labels = len(list(labels_path.glob('*.txt')))
        print(f"  {split}/labels: {num_labels} labels")

print(f"\nTotal images: {total_images}")

In [ ]:
# Create a directory with all images for distillation (unlabeled pretraining)
UNLABELED_DIR = Path("./unlabeled_images")
UNLABELED_DIR.mkdir(exist_ok=True)

total_copied = 0
for split in ['train', 'valid', 'test']:
    images_path = dataset_path / split / 'images'
    if images_path.exists():
        for img_file in images_path.glob('*'):
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
                dest = UNLABELED_DIR / img_file.name
                if not dest.exists():
                    shutil.copy2(img_file, dest)
                    total_copied += 1

print(f"Copied {total_copied} images")
print(f"Total images for distillation: {len(list(UNLABELED_DIR.glob('*')))}")

## 4. DINOv3 Distillation Pretraining

Pretrain the YOLO11l-seg backbone using knowledge distillation from DINOv3.

In [ ]:
# Configure distillation parameters
import lightly_train

OUTPUT_DIR = "./output/dinov3_yolo11l_distill"

distill_config = {
    "out": OUTPUT_DIR,
    "data": str(UNLABELED_DIR),
    "model": STUDENT_MODEL,
    "method": "distillation",
    "method_args": {"teacher": TEACHER_MODEL},
    "epochs": DISTILL_EPOCHS,
    "batch_size": DISTILL_BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "precision": PRECISION,
}

if DISTILL_LR is not None:
    distill_config["lr"] = DISTILL_LR

print("Distillation Configuration:")
for k, v in distill_config.items():
    print(f"  {k}: {v}")

In [ ]:
# Run distillation pretraining
print("Starting DINOv3 -> YOLO11l-seg distillation...")
print(f"Using {len(list(UNLABELED_DIR.glob('*')))} unlabeled images")
print("="*60)

lightly_train.pretrain(**distill_config)

print("="*60)
print("Distillation complete!")

### 4.1 Distillation Training History

In [ ]:
# Plot distillation training history
distill_log_path = Path(OUTPUT_DIR) / "metrics.csv"

if distill_log_path.exists():
    distill_df = pd.read_csv(distill_log_path)
    distill_df.columns = distill_df.columns.str.strip()
    
    print(f"Distillation metrics columns: {list(distill_df.columns)}")
    
    # Find loss columns
    loss_cols = [c for c in distill_df.columns if 'loss' in c.lower()]
    
    if loss_cols:
        fig, ax = plt.subplots(figsize=(10, 5))
        
        for i, col in enumerate(loss_cols):
            color = list(COLORS.values())[i % len(COLORS)]
            ax.plot(distill_df[col], label=col, color=color, linewidth=2)
        
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title('DINOv3 Distillation - Training Loss', fontweight='bold', fontsize=14)
        ax.legend(frameon=False)
        plt.tight_layout()
        plt.show()
    else:
        print("No loss columns found in distillation metrics")
else:
    print(f"Distillation metrics file not found at {distill_log_path}")
    # Try alternative paths
    for f in Path(OUTPUT_DIR).rglob('*.csv'):
        print(f"  Found: {f}")

## 5. Fine-tune for Instance Segmentation

Load the pretrained backbone and fine-tune on the labeled TACO dataset.

In [ ]:
# Path to the pretrained model
PRETRAINED_MODEL_PATH = Path(OUTPUT_DIR) / "exported_models" / "exported_last.pt"

if not PRETRAINED_MODEL_PATH.exists():
    possible_paths = list(Path(OUTPUT_DIR).rglob("*.pt"))
    print("Available .pt files:")
    for p in possible_paths:
        print(f"  {p}")
    if possible_paths:
        PRETRAINED_MODEL_PATH = possible_paths[0]

print(f"Using pretrained model: {PRETRAINED_MODEL_PATH}")

In [ ]:
# Load the pretrained model
from ultralytics import YOLO

model = YOLO(str(PRETRAINED_MODEL_PATH))
print(f"Model loaded: {model.model.__class__.__name__}")
print(f"Task: {model.task}")

In [ ]:
# Fine-tune for instance segmentation
DATA_YAML = dataset_path / "data.yaml"
FINETUNE_OUTPUT = "./output/finetune_seg"

results = model.train(
    data=str(DATA_YAML),
    epochs=FINETUNE_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=FINETUNE_BATCH_SIZE,
    lr0=FINETUNE_LR,
    lrf=FINETUNE_LRF,
    workers=NUM_WORKERS,
    project=FINETUNE_OUTPUT,
    name="yolo11l_seg_taco",
    patience=PATIENCE,
    save=True,
    plots=True,
)

### 5.1 Fine-tuning Training History

In [ ]:
# Plot fine-tuning training history
finetune_csv = Path(FINETUNE_OUTPUT) / "yolo11l_seg_taco" / "results.csv"

if finetune_csv.exists():
    ft_df = pd.read_csv(finetune_csv)
    ft_df.columns = ft_df.columns.str.strip()
    
    # Plot losses
    loss_cols = ['train/box_loss', 'train/seg_loss', 'train/cls_loss', 'train/dfl_loss']
    available_loss = [c for c in loss_cols if c in ft_df.columns]
    
    if available_loss:
        fig, axes = plt.subplots(1, len(available_loss), figsize=(4*len(available_loss), 4))
        if len(available_loss) == 1:
            axes = [axes]
        
        for ax, col in zip(axes, available_loss):
            ax.plot(ft_df[col], color=COLORS['blue'], linewidth=2)
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.set_title(col.split('/')[-1].replace('_', ' ').title(), fontweight='bold')
        
        plt.suptitle('Fine-tuning - Training Losses', fontweight='bold', fontsize=14, y=1.02)
        plt.tight_layout()
        plt.show()
    
    # Plot mAP curves
    map_cols = ['metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/mAP50(M)', 'metrics/mAP50-95(M)']
    available_map = [c for c in map_cols if c in ft_df.columns]
    
    if available_map:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        # Box mAP
        ax = axes[0]
        if 'metrics/mAP50(B)' in ft_df.columns:
            ax.plot(ft_df['metrics/mAP50(B)'], color=COLORS['blue'], linewidth=2, label='mAP50')
        if 'metrics/mAP50-95(B)' in ft_df.columns:
            ax.plot(ft_df['metrics/mAP50-95(B)'], color=COLORS['orange'], linewidth=2, label='mAP50-95')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('mAP')
        ax.set_title('Box Detection mAP', fontweight='bold')
        ax.legend(frameon=False)
        
        # Mask mAP
        ax = axes[1]
        if 'metrics/mAP50(M)' in ft_df.columns:
            ax.plot(ft_df['metrics/mAP50(M)'], color=COLORS['green'], linewidth=2, label='mAP50')
        if 'metrics/mAP50-95(M)' in ft_df.columns:
            ax.plot(ft_df['metrics/mAP50-95(M)'], color=COLORS['red'], linewidth=2, label='mAP50-95')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('mAP')
        ax.set_title('Segmentation mAP', fontweight='bold')
        ax.legend(frameon=False)
        
        plt.suptitle('Fine-tuning - Validation mAP', fontweight='bold', fontsize=14, y=1.02)
        plt.tight_layout()
        plt.show()
else:
    print(f"Fine-tuning results not found at {finetune_csv}")

## 6. Evaluation & Results

In [ ]:
# Validate the fine-tuned model
metrics = model.val()

# Display results nicely
print("\n" + "="*60)
print("       RESULTS: DINOv3 Distillation + Fine-tuning")
print("="*60)
print(f"\n  {'Metric':<25} {'Value':>10}")
print(f"  {'-'*35}")
print(f"  {'Box mAP50':<25} {metrics.box.map50:>10.4f}")
print(f"  {'Box mAP50-95':<25} {metrics.box.map:>10.4f}")
print(f"  {'Mask mAP50':<25} {metrics.seg.map50:>10.4f}")
print(f"  {'Mask mAP50-95':<25} {metrics.seg.map:>10.4f}")
print("\n" + "="*60)

In [ ]:
# Visual metrics summary
fig, ax = plt.subplots(figsize=(8, 5))

metrics_names = ['Box\nmAP50', 'Box\nmAP50-95', 'Mask\nmAP50', 'Mask\nmAP50-95']
metrics_values = [metrics.box.map50, metrics.box.map, metrics.seg.map50, metrics.seg.map]
colors = [COLORS['blue'], COLORS['blue'], COLORS['green'], COLORS['green']]
alphas = [1.0, 0.6, 1.0, 0.6]

bars = ax.bar(metrics_names, metrics_values, color=colors, alpha=alphas, edgecolor='white', linewidth=2)

for bar, val in zip(bars, metrics_values):
    ax.annotate(f'{val:.4f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 5), textcoords='offset points', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('mAP Score', fontsize=12)
ax.set_title('DINOv3 Distillation Results', fontweight='bold', fontsize=14)
ax.set_ylim(0, max(metrics_values) * 1.2)
plt.tight_layout()
plt.show()

## 7. Inference Demo

In [ ]:
# Run inference on test images with polygon masks
from PIL import Image
import random

test_images_path = dataset_path / "test" / "images"
test_images = list(test_images_path.glob("*.jpg")) + list(test_images_path.glob("*.JPG"))
random.shuffle(test_images)
test_images = test_images[:6]

print(f"Running inference on {len(test_images)} test images...")

In [ ]:
# Visualize predictions with polygon segmentation masks
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for ax, img_path in zip(axes, test_images):
    # Run prediction
    results = model.predict(str(img_path), verbose=False)
    
    # Plot with masks, boxes, and labels
    result_img = results[0].plot(
        masks=True,           # Show segmentation masks
        boxes=True,           # Show bounding boxes
        labels=True,          # Show class labels
        conf=True,            # Show confidence scores
        line_width=2,
    )
    
    ax.imshow(result_img[:, :, ::-1])
    ax.axis('off')
    
    # Count detections
    n_detections = len(results[0].boxes) if results[0].boxes is not None else 0
    ax.set_title(f'{img_path.name[:25]}...\n{n_detections} detections', fontsize=10)

plt.suptitle('YOLO11l-seg (DINOv3 Distillation) - Predictions with Polygon Masks', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Save & Export

In [ ]:
# Save metrics to JSON
import json

Path(FINETUNE_OUTPUT).mkdir(parents=True, exist_ok=True)

metrics_dict = {
    "model": "YOLO11l-seg",
    "training": "dinov3_distillation",
    "hyperparameters": {
        "distill_epochs": DISTILL_EPOCHS,
        "distill_batch_size": DISTILL_BATCH_SIZE,
        "finetune_epochs": FINETUNE_EPOCHS,
        "finetune_batch_size": FINETUNE_BATCH_SIZE,
        "finetune_lr": FINETUNE_LR,
        "image_size": IMAGE_SIZE,
    },
    "results": {
        "box_map50": float(metrics.box.map50),
        "box_map": float(metrics.box.map),
        "seg_map50": float(metrics.seg.map50),
        "seg_map": float(metrics.seg.map),
    }
}

with open(f"{FINETUNE_OUTPUT}/metrics_distillation.json", "w") as f:
    json.dump(metrics_dict, f, indent=2)

print(f"Metrics saved to: {FINETUNE_OUTPUT}/metrics_distillation.json")

In [ ]:
# Export to ONNX
model.export(format="onnx", imgsz=IMAGE_SIZE, simplify=True)
print("Model exported to ONNX format")

In [ ]:
# Download the trained model (for Colab)
try:
    from google.colab import files
    
    best_model_path = Path(FINETUNE_OUTPUT) / "yolo11l_seg_taco" / "weights" / "best.pt"
    
    if best_model_path.exists():
        print(f"Downloading {best_model_path}...")
        files.download(str(best_model_path))
    else:
        print("Best model not found. Available files:")
        for f in Path(FINETUNE_OUTPUT).rglob("*.pt"):
            print(f"  {f}")
except ImportError:
    print("Not running in Colab - skipping download")

## Summary

This notebook trained **YOLO11l-seg with DINOv3 distillation pretraining** on the TACO dataset.

**Two-stage training:**
1. **Distillation**: Transfer knowledge from DINOv3 → YOLO11l-seg backbone (unlabeled)
2. **Fine-tuning**: Train on labeled instance segmentation data

**Compare with:** `train_from_scratch.ipynb` to see the benefit of distillation pretraining.

**Expected benefit:** Better mAP scores, faster convergence, better generalization on small datasets.

**Dataset:** TACO (Trash Annotations in Context)
- Paper: [arXiv:2003.06975](https://arxiv.org/abs/2003.06975)
- License: CC BY 4.0